# German FrameParser — end-to-end inference

Runs the full chain on raw German text: **trigger lexicon-rule → frame model → argument model**, loading your two trained checkpoints from Drive.

### Before you run
1. **Re-make the code zip so it includes `salsa_pipeline.py`** (the zip glob `German_parser/*.py` picks it up automatically — just re-run the same command and re-upload):
```bash
cd ~/Desktop/Academic_projects/Texture_Frames
zip -r texture_frames_colab.zip \
    encoder_parser German_parser/*.py \
    German_parser/extracted/salsa_release.xml \
    German_parser/extracted/salsa_frames.xml
```
2. Your trained models must be on Drive at `MyDrive/Texture_Frames/models/frame2_de` and `.../args2_de` (the train notebooks put them there).

GPU is optional — inference runs on CPU too, just slower. `Runtime → GPU` recommended.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
ZIP = "/content/drive/MyDrive/texture_frames_colab.zip"
assert os.path.exists(ZIP), f"Upload the (re-made) project zip to {ZIP}."
!rm -rf /content/Texture_Frames && mkdir -p /content/Texture_Frames
!unzip -q "$ZIP" -d /content/Texture_Frames

base = "/content/Texture_Frames"
FRAME_DIR = "/content/drive/MyDrive/Texture_Frames/models/frame2_de"
ARGS_DIR  = "/content/drive/MyDrive/Texture_Frames/models/args2_de"
need = {
    "salsa_pipeline.py"        : os.path.join(base, "German_parser/salsa_pipeline.py"),
    "SALSA corpus"             : os.path.join(base, "German_parser/extracted/salsa_release.xml"),
    "frame checkpoint (Drive)" : os.path.join(FRAME_DIR, "frame2_model.pt"),
    "args checkpoint (Drive)"  : os.path.join(ARGS_DIR,  "args2_model.pt"),
}
for name, path in need.items():
    print(("OK  " if os.path.exists(path) else "MISSING  "), name)
assert all(os.path.exists(p) for p in need.values()), "Fix the missing item(s) above."

In [ ]:
!pip install -q "transformers==4.57.6" "huggingface_hub<1.0" sentencepiece simplemma "numpy>=2"

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")
sys.path.insert(0, "/content/Texture_Frames/German_parser")
os.chdir("/content/Texture_Frames/German_parser")
import torch, transformers
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
from huggingface_hub import __version__ as _hv
assert int(_hv.split(".")[0]) < 1, (
    f"huggingface_hub {_hv} is loaded, but transformers 4.57.6 needs <1.0 \u2014 "
    "do Runtime \u2192 Restart session, then Run all from the top.")

## Load the parser

First run takes ~1–2 min: it builds the SALSA lexicon + trigger table from the corpus and loads both gbert checkpoints. The operating points default to the dev-picked values (frame bias 4.0, args NULL-bias 2.0).

In [ ]:
from salsa_pipeline import GermanFrameParser

parser = GermanFrameParser(frame_dir=FRAME_DIR, args_dir=ARGS_DIR)
print("parser ready — device:", parser.device)

## Parse example sentences

In [ ]:
import dataclasses

examples = [
    "Die Regierung kündigte an , die Steuern zu erhöhen .",
    "Der Vorstand ernannte sie zur Vorsitzenden .",
    "Die Polizei verhaftete den Verdächtigen am Bahnhof .",
]
for sent in examples:
    print("#", sent)
    anns = parser.parse(sent)
    if not anns:
        print("   (no triggers detected)")
    for ann in anns:
        print(f"  [{ann.frame}]  trigger={ann.trigger!r}")
        for a in ann.arguments:
            print(f"      {a.role:16} {a.text!r}")
    print()

## Parse your own text

Edit `sentence` below. Note: the trigger rule and gold data are whitespace-tokenized, so separate punctuation with spaces (e.g. `… erhöhen .`) for best alignment.

In [ ]:
sentence = "Der Minister erklärte den Vertrag für ungültig ."

for ann in parser.parse(sentence):
    print(f"[{ann.frame}]  trigger={ann.trigger!r}  (@{ann.trigger_loc})")
    for a in ann.arguments:
        print(f"    {a.role:16} {a.text!r}  (chars {a.start}:{a.end})")

## Notes

- **Trigger stage is the lexicon rule** (`salsa_trigger`, F1≈0.88) — it fires only on SALSA's ~665 covered content lemmas; that's the honest coverage ceiling of a SALSA-trained detector.
- Tune operating points at construction: `GermanFrameParser(..., frame_bias=4.0, null_bias=2.0, trigger_threshold=0.5)`. Lower `trigger_threshold` for more (but less precise) triggers; raise `null_bias` for fewer, more precise arguments.
- To get JSON: `import dataclasses, json; json.dumps([dataclasses.asdict(a) for a in parser.parse(text)])`.